# AlphaLens XGBoost Experiment

This notebook selects a regularized XGBoost regressor using only the purged chronological validation period. The selected parameters and validation-selected tree count are refit on train plus validation, then compared once with the locked test baselines.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.features import build_event_feature_dataset
from pipelines.ml.xgboost_model import run_xgboost_experiment

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

## Validation-only model selection

In [ ]:
dataset = build_event_feature_dataset(horizon=30, include_topics=True)
experiment = run_xgboost_experiment(dataset)
display(pd.Series(experiment.selected_parameters, name='value'))
print('Selected boosting rounds:', experiment.selected_boosting_rounds)

In [ ]:
validation = experiment.validation_metrics.drop(columns=['parameters']).copy()
display(validation[['model', 'mae', 'rmse', 'directional_accuracy', 'pearson', 'spearman_ic', 'boosting_rounds']])

## Untouched test comparison

The test period does not choose parameters, tree count, or features. It answers only whether the already-selected model generalizes better than the simple hurdles.

In [ ]:
test_metrics = experiment.test_metrics.drop(columns=['parameters']).copy()
display(test_metrics[['model', 'mae', 'rmse', 'r2', 'directional_accuracy', 'pearson', 'spearman_ic']])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=test_metrics, x='mae', y='model', ax=axes[0], color='#2563eb')
sns.barplot(data=test_metrics, x='rmse', y='model', ax=axes[1], color='#0f766e')
axes[0].set(title='Untouched test MAE', xlabel='Lower is better', ylabel='')
axes[1].set(title='Untouched test RMSE', xlabel='Lower is better', ylabel='')
plt.tight_layout()
plt.show()

In [ ]:
xgb_predictions = experiment.test_predictions.loc[
    experiment.test_predictions['model'].str.startswith('xgboost')
].copy()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.scatterplot(data=xgb_predictions, x='prediction', y='excess_return_30d', hue='event_source', ax=ax)
limits = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(limits, limits, linestyle='--', color='#111827', linewidth=1)
ax.set(title='XGBoost predictions versus realized excess returns', xlabel='Predicted 30-session excess return', ylabel='Realized 30-session excess return')
plt.tight_layout()
plt.show()

## Decision rule

A complex model is not promoted merely because it fits. It must improve untouched test MAE or RMSE and show credible directional or rank behavior. A near miss is retained as an experiment result, while the historical-mean predictor remains the deployment hurdle.